# Project: Fine-Tuning Your Own LLaMA 2 Model
**Module 06 – Developing Large Language Models**

> *Reference articles included in this module:*
> - Introduction to Meta AI's LLaMA
> - LLaMA.cpp Tutorial: Efficient LLM Inference
> - What is LLaMA 3? The Next Generation of Open Source LLMs

## What is LLaMA?

**LLaMA (Large Language Model Meta AI)** is Meta's open-source family of foundation language models:

| Model | Parameters | Key Features |
|---|---|---|
| LLaMA 1 | 7B–65B | First open release from Meta |
| LLaMA 2 | 7B–70B | Improved, commercial license for many uses |
| LLaMA 3 | 8B–70B+ | State-of-the-art open-source performance |

LLaMA models power many open-source AI applications because:
- Weights are publicly available
- Can be fine-tuned for custom tasks
- Run efficiently on consumer hardware with quantization

## Fine-Tuning Strategy: Parameter-Efficient Fine-Tuning (PEFT)

Fine-tuning a full 7B+ parameter model requires enormous compute. We use **PEFT techniques**:

| Technique | Description |
|---|---|
| **LoRA** | Low-Rank Adaptation — adds small trainable matrices |
| **QLoRA** | Quantized LoRA — enables 4-bit quantization + LoRA |
| **Adapters** | Small modules inserted between frozen layers |

In [ ]:
# Required libraries
# pip install transformers peft accelerate bitsandbytes datasets trl

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

print("Libraries imported successfully!")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1: Load LLaMA 2 with 4-bit Quantization (QLoRA)

In [ ]:
# Quantization configuration (4-bit for memory efficiency)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True     # Double quantization for more savings
)

# Load model (requires Hugging Face access to LLaMA 2)
# model_name = "meta-llama/Llama-2-7b-hf"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# model = prepare_model_for_kbit_training(model)

print("[Simulated] LLaMA 2 loaded in 4-bit precision")

## Step 2: Configure LoRA Adapters

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                         # Rank of the low-rank matrices
    lora_alpha=32,                # Scaling factor
    target_modules=[              # Which layers to apply LoRA to
        "q_proj", "v_proj",
        "k_proj", "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the model
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()
# Example output: trainable params: 4,194,304 || all params: 6,742,609,920 (0.06%)
print("LoRA config:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")

## Step 3: Prepare Dataset and Fine-Tune

In [ ]:
# Sample instruction-tuning dataset
train_data = [
    {"text": "<s>[INST] What is deep learning? [/INST] Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn representations from data. </s>"},
    {"text": "<s>[INST] What is a transformer? [/INST] A transformer is a neural network architecture that uses self-attention mechanisms to process sequences in parallel, enabling better handling of long-range dependencies. </s>"},
    {"text": "<s>[INST] Explain LLaMA in simple terms. [/INST] LLaMA is Meta's open-source family of large language models. They can understand and generate text, and can be fine-tuned for specific tasks. </s>"},
]

dataset = Dataset.from_list(train_data)
print("Training dataset:")
print(dataset)

# Training arguments
training_args = TrainingArguments(
    output_dir="./llama2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

print("\nTraining configuration ready!")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

## Step 4: Running Inference with LLaMA.cpp

For local inference without a GPU, **llama.cpp** provides quantized GGUF format models.

In [ ]:
# Install: pip install llama-cpp-python

# from llama_cpp import Llama

# Load a GGUF quantized model
# llm = Llama(
#     model_path="./models/llama-2-7b-chat.Q4_K_M.gguf",
#     n_ctx=2048,       # context window
#     n_threads=8,      # CPU threads
# )

# Inference
# output = llm(
#     "Q: What is machine learning? A:",
#     max_tokens=100,
#     stop=["Q:", "\n"],
#     echo=True
# )
# print(output['choices'][0]['text'])

print("""LLaMA.cpp workflow:
1. Download quantized model (.gguf format)
2. Load with Llama() class
3. Run inference locally on CPU or GPU
4. No internet connection required after download""")

## LLaMA 3 Improvements

LLaMA 3 (2024) brought significant improvements:

| Feature | LLaMA 2 | LLaMA 3 |
|---|---|---|
| Vocabulary | 32k tokens | 128k tokens |
| Context window | 4k tokens | 8k tokens (base), 128k+ (extended) |
| Architecture | Standard transformer | Grouped Query Attention (GQA) |
| Performance | Strong | State-of-the-art among open models |
| Multilingual | Limited | Significantly improved |

## Key Takeaways

- **QLoRA** makes fine-tuning LLaMA 2 accessible on consumer GPUs
- Only ~0.06% of parameters need to be trained with LoRA
- **llama.cpp** enables efficient CPU inference with quantization
- LLaMA 3 represents a major leap in open-source LLM capability